In [0]:
SILVER = "workspace.olist_silver"
GOLD = "workspace.olist_gold"


def criar(tabela, consulta):
    """Roda a consulta SQL, grava o resultado na camada ouro e mostra a contagem."""
    df = spark.sql(consulta)
    df.write.mode("overwrite").saveAsTable(f"{GOLD}.{tabela}")
    linhas = spark.table(f"{GOLD}.{tabela}").count()
    print(f"✅ {tabela}: {linhas} linhas")

In [0]:
criar("dim_products", f"""
SELECT
  product_id,
  product_category_name          AS category_pt,
  product_category_name_english  AS category_en,
  product_weight_g               AS weight_g,
  product_length_cm              AS length_cm,
  product_height_cm              AS height_cm,
  product_width_cm               AS width_cm,
  product_photos_qty             AS photos_qty
FROM {SILVER}.products
""")

✅ dim_products: 32951 linhas


In [0]:
criar("dim_customers", f"""
SELECT
  c.customer_id,
  c.customer_unique_id,
  c.customer_zip_code_prefix AS zip_code_prefix,
  c.customer_city            AS city,
  c.customer_state           AS state,
  CASE
    WHEN c.customer_state IN ('AC','AP','AM','PA','RO','RR','TO') THEN 'Norte'
    WHEN c.customer_state IN ('AL','BA','CE','MA','PB','PE','PI','RN','SE') THEN 'Nordeste'
    WHEN c.customer_state IN ('DF','GO','MT','MS') THEN 'Centro-Oeste'
    WHEN c.customer_state IN ('ES','MG','RJ','SP') THEN 'Sudeste'
    WHEN c.customer_state IN ('PR','RS','SC') THEN 'Sul'
    ELSE 'Não informado'
  END AS region,
  g.geolocation_lat AS latitude,
  g.geolocation_lng AS longitude
FROM {SILVER}.customers AS c
LEFT JOIN {SILVER}.geolocation AS g
  ON c.customer_zip_code_prefix = g.geolocation_zip_code_prefix
""")

✅ dim_customers: 99441 linhas


In [0]:
criar("dim_sellers", f"""
SELECT
  s.seller_id,
  s.seller_zip_code_prefix AS zip_code_prefix,
  s.seller_city            AS city,
  s.seller_state           AS state,
  CASE
    WHEN s.seller_state IN ('AC','AP','AM','PA','RO','RR','TO') THEN 'Norte'
    WHEN s.seller_state IN ('AL','BA','CE','MA','PB','PE','PI','RN','SE') THEN 'Nordeste'
    WHEN s.seller_state IN ('DF','GO','MT','MS') THEN 'Centro-Oeste'
    WHEN s.seller_state IN ('ES','MG','RJ','SP') THEN 'Sudeste'
    WHEN s.seller_state IN ('PR','RS','SC') THEN 'Sul'
    ELSE 'Não informado'
  END AS region,
  g.geolocation_lat AS latitude,
  g.geolocation_lng AS longitude
FROM {SILVER}.sellers AS s
LEFT JOIN {SILVER}.geolocation AS g
  ON s.seller_zip_code_prefix = g.geolocation_zip_code_prefix
""")

✅ dim_sellers: 3095 linhas


In [0]:
criar("dim_date", """
SELECT
  d                       AS full_date,
  year(d)                 AS year,
  quarter(d)              AS quarter,
  month(d)                AS month,
  date_format(d, 'yyyy-MM') AS year_month,
  CASE month(d)
    WHEN 1 THEN 'Janeiro'   WHEN 2 THEN 'Fevereiro' WHEN 3 THEN 'Março'
    WHEN 4 THEN 'Abril'     WHEN 5 THEN 'Maio'      WHEN 6 THEN 'Junho'
    WHEN 7 THEN 'Julho'     WHEN 8 THEN 'Agosto'    WHEN 9 THEN 'Setembro'
    WHEN 10 THEN 'Outubro'  WHEN 11 THEN 'Novembro' WHEN 12 THEN 'Dezembro'
  END                     AS month_name,
  dayofweek(d)            AS day_of_week,
  CASE dayofweek(d)
    WHEN 1 THEN 'Domingo'  WHEN 2 THEN 'Segunda-feira' WHEN 3 THEN 'Terça-feira'
    WHEN 4 THEN 'Quarta-feira' WHEN 5 THEN 'Quinta-feira'
    WHEN 6 THEN 'Sexta-feira'  WHEN 7 THEN 'Sábado'
  END                     AS day_name,
  dayofweek(d) IN (1, 7)  AS is_weekend
FROM (
  SELECT explode(sequence(to_date('2016-01-01'), to_date('2018-12-31'), interval 1 day)) AS d
)
""")

✅ dim_date: 1096 linhas


In [0]:
criar("agg_order_payments", f"""
SELECT
  order_id,
  ROUND(SUM(payment_value), 2)  AS total_paid,
  COUNT(*)                      AS payment_count,
  MAX(payment_installments)     AS max_installments,
  max_by(payment_type, struct(payment_value, -payment_sequential)) AS main_payment_type,
  bool_or(payment_type = 'voucher') AS has_voucher
FROM {SILVER}.order_payments
GROUP BY order_id
""")

✅ agg_order_payments: 99440 linhas


In [0]:
criar("fact_order_items", f"""
SELECT
  i.order_id,
  i.order_item_id,
  i.product_id,
  i.seller_id,
  o.customer_id,
  to_date(o.order_purchase_timestamp)         AS purchase_date,
  to_date(o.order_delivered_customer_date)    AS delivered_date,
  to_date(o.order_estimated_delivery_date)    AS estimated_delivery_date,
  o.order_status,
  i.price,
  i.freight_value,
  ROUND(i.price + i.freight_value, 2)         AS item_total,
  datediff(to_date(o.order_delivered_customer_date),
           to_date(o.order_purchase_timestamp))        AS delivery_days,
  datediff(to_date(o.order_delivered_customer_date),
           to_date(o.order_estimated_delivery_date))   AS delay_days,
  to_date(o.order_delivered_customer_date) > to_date(o.order_estimated_delivery_date) AS is_late,
  p.main_payment_type,
  p.has_voucher,
  r.review_score
FROM {SILVER}.order_items AS i
INNER JOIN {SILVER}.orders AS o
  ON i.order_id = o.order_id
LEFT JOIN {GOLD}.agg_order_payments AS p
  ON i.order_id = p.order_id
LEFT JOIN {SILVER}.order_reviews AS r
  ON i.order_id = r.order_id
""")

✅ fact_order_items: 112650 linhas
